# Used Packages
- **`transformers`** – Hugging Face library for loading and fine-tuning pre-trained models (AraBERT, mT5)
- **`datasets`** – Hugging Face library for handling and processing datasets efficiently
- **`accelerate`** – Enables faster, distributed training on GPU
- **`arabic-reshaper`** & **`python-bidi`** – Handle proper rendering and reshaping of Arabic text
- **`scikit-learn`** – Used for label encoding, train/test splits, and evaluation metrics

In [ ]:
!pip install -q transformers datasets accelerate arabic-reshaper python-bidi scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 20.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile

# Base directory to search in Google Drive
base_dir = "/content/drive/MyDrive"

# Automatically search for any SauDial*.zip inside MyDrive
zip_path = None
for root, dirs, files in os.walk(base_dir):
    for f in files:
        if "SauDial" in f and f.lower().endswith(".zip"):
            zip_path = os.path.join(root, f)
            break
    if zip_path:
        break

if zip_path is None:
    print("SauDial ZIP file not found. Please upload or add a shortcut to MyDrive.")
else:
    print("SauDial ZIP file detected at:")
    print(zip_path)

    # Extract to content folder
    extract_path = "/content/SauDial"
    os.makedirs(extract_path, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    print("\nExtracted directory structure:\n")
    for root, dirs, files in os.walk(extract_path):
        level = root.replace(extract_path, "").count(os.sep)
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = "  " * (level + 1)
        for f in files:
            print(f"{subindent}{f}")


Mounted at /content/drive
SauDial ZIP file detected at:
/content/drive/MyDrive/NLP Project/Datasets/SauDial The Saudi Arabic Dialects Game Localizatio.zip

Extracted directory structure:

SauDial/
  SauDial The Saudi Arabic Dialects Game Localizatio/
    saudial_generator.py
    SauDial Dataset.xlsx


# Loading and Previewing the Raw Dataset

Read the SauDial Excel file into a **pandas DataFrame** using `pd.read_excel()` and displays the first 5 rows.

At this stage, the DataFrame contains **all available columns**, including:
- `Dialect` — the regional dialect label (Najdi, Hijazi, Eastern, Southern)
- `Dialect Translation` — the text written in that dialect
- `Modern Standard Arabic (MSA) Translation` — the corresponding MSA version of the text


In [ ]:
import pandas as pd

df = pd.read_excel("/content/SauDial/SauDial The Saudi Arabic Dialects Game Localizatio/SauDial Dataset.xlsx")
df.head()


,Dialect,Scenario,Game Type,Tone,Age Rating,English Text,Modern Standard Arabic (MSA) Translation,Dialect Translation,Context and Rating,Dialect Notes,Localization Difficulty,In-Game Context
0,Najdi,Holographic Desert Racing,Action,Excited,12+,"""Rev up your engines! Today's race takes you t...","""شغلوا محركاتكم! سباق اليوم يأخذكم عبر طرق الت...","""قوّوا مكاينكم! سباق اليوم بياخذكم على دروب ال...",This dialogue combines traditional settings wi...,"""بياخذكم"" is used for ""takes you"", and ""دروب"" ...",4,Player participating in a high-tech desert rac...
1,Hijazi,Jeddah Tower Escape Room,Puzzle,Mysterious,18+,"""You're trapped on the 300th floor of Jeddah T...","""أنت محاصر في الطابق الثلاثمائة من برج جدة. حل...","""انت محبوس في الدور الـ300 من برج جدة. حل الأل...",This dialogue mixes modern architecture with t...,"""محبوس"" is used for ""trapped"", and ""بينزلوا"" f...",4,Player solving puzzles to escape from a futuri...
2,Eastern,Dhow Building Simulator,Educational,Reflective,3+,"""Grandpa says, 'A well-built dhow can weather ...","""يقول جدي: 'السفينة المبنية جيدًا يمكنها مواجه...","""جدي يقول: 'البوم المضبوط يقدر يواجه أي نوة.' ...",This dialogue teaches traditional boat-buildin...,"""مضبوط"" is used for ""well-built"", and ""نوة"" fo...",5,Player learning to build a traditional dhow boat.
3,Southern,Mountain Terrace Farming,Simulation,Serious,12+,"""The rains are coming early this year. We need...","""الأمطار تأتي مبكرًا هذا العام. نحتاج إلى تقوي...","""المطر جاي بدري السنة. لازم نقوي المساطب ونزرع...",This dialogue addresses climate change adaptat...,"""مساطب"" is used for terraces, and ""بنتأقلم"" fo...",4,Player managing a traditional mountain farm fa...
4,Najdi,Camel Beauty Pageant,Role-Playing,Humorous,3+,"""Welcome to the Royal Camel Beauty Pageant! Ju...","""مرحبًا بكم في مسابقة جمال الإبل الملكية! احكم...","""حياكم في مسابقة زين الهجن الملكية! حكموا على ...",This humorous dialogue plays on traditional ca...,"""زين"" is used for beauty, and ""غوارب"" for hump...",3,Player judging a humorous camel beauty contest.


#**Classification**
Saudi Dialect Classification

1. **Select only the two relevant columns**: `"Dialect"` (label) and `"Dialect Translation"` (input text)
2. **Drop any rows with missing values** using `.dropna()` to ensure a clean, complete dataset

The resulting `df_cls` DataFrame contains only what is needed for training the classifier — raw dialect text alongside its regional label.

We also print a **class distribution** using `.value_counts()` to verify that the four dialect classes are reasonably balanced, which is important for reliable classifier training.

In [ ]:

import pandas as pd

# Load the dataset
excel_path = "/content/SauDial/SauDial The Saudi Arabic Dialects Game Localizatio/SauDial Dataset.xlsx"
df = pd.read_excel(excel_path)

# Keep only columns needed for classification
df_cls = df[["Dialect", "Dialect Translation"]].dropna()

print(df_cls.head())
print(df_cls["Dialect"].value_counts())


    Dialect                                Dialect Translation
0     Najdi  "قوّوا مكاينكم! سباق اليوم بياخذكم على دروب ال...
1    Hijazi  "انت محبوس في الدور الـ300 من برج جدة. حل الأل...
2   Eastern  "جدي يقول: 'البوم المضبوط يقدر يواجه أي نوة.' ...
3  Southern  "المطر جاي بدري السنة. لازم نقوي المساطب ونزرع...
4     Najdi  "حياكم في مسابقة زين الهجن الملكية! حكموا على ...
Dialect
Najdi       271
Hijazi      265
Eastern     241
Southern    223
Name: count, dtype: int64


#Text cleaning & Pre-Prosessing

Before feeding text into a language model, we must **normalise and clean** the Arabic dialect data. The `clean_text()` function applies the following steps to each text sample:

| Step | What it does |
|---|---|
| **Remove diacritics** | Strips Arabic tashkeel (short vowel marks: Unicode `\u064B–\u0652`) which add noise without changing word identity |
| **Keep Arabic only** | Removes any non-Arabic characters (numbers, punctuation, Latin letters) using a Unicode range filter |
| **Normalize whitespace** | Collapses multiple spaces into one and strips leading/trailing whitespace |

The cleaned text is stored in a new column called **`clean_text`**, which becomes the primary model input.

Rows with empty strings after cleaning are also removed to avoid training on meaningless samples.

In [ ]:
import re

def clean_text(text):
    text = str(text)
    text = re.sub(r'[\u064B-\u0652]', '', text)   # remove diacritics
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)  # keep Arabic only
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_cls["clean_text"] = df_cls["Dialect Translation"].apply(clean_text)
df_cls = df_cls[df_cls["clean_text"].str.len() > 0]

df_cls.head()


,Dialect,Dialect Translation,clean_text
0,Najdi,"""قوّوا مكاينكم! سباق اليوم بياخذكم على دروب ال...",قووا مكاينكم سباق اليوم بياخذكم على دروب التجا...
1,Hijazi,"""انت محبوس في الدور الـ300 من برج جدة. حل الأل...",انت محبوس في الدور الـ من برج جدة حل الألغاز ع...
2,Eastern,"""جدي يقول: 'البوم المضبوط يقدر يواجه أي نوة.' ...",جدي يقول البوم المضبوط يقدر يواجه أي نوة يلا ن...
3,Southern,"""المطر جاي بدري السنة. لازم نقوي المساطب ونزرع...",المطر جاي بدري السنة لازم نقوي المساطب ونزرع م...
4,Najdi,"""حياكم في مسابقة زين الهجن الملكية! حكموا على ...",حياكم في مسابقة زين الهجن الملكية حكموا على ال...


# Label Encoding and Dataset Splitting

### - Label Encoding
Since machine learning models require numerical targets, we use **`LabelEncoder`** from scikit-learn to convert dialect names into integer class IDs

### - Dataset Split
The dataset is divided into three non-overlapping subsets using a stratified split — meaning each dialect class maintains proportional representation in all splits:

- **80% Training** — used to update model weights
- **10% Validation** — used to monitor performance after each epoch and prevent overfitting
- **10% Test** — held-out set used only for final evaluation; never seen during training

Stratification ensures that even smaller dialect classes are fairly represented across all splits.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Encode dialect labels as integers
label_encoder = LabelEncoder()
df_cls["label"] = label_encoder.fit_transform(df_cls["Dialect"])

print("Label mapping (class -> id):")
for cls, idx in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"{cls} -> {idx}")

# Train/validation/test split: 80% train, 10% val, 10% test (stratified)
train_df, temp_df = train_test_split(
    df_cls,
    test_size=0.2,
    random_state=42,
    stratify=df_cls["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

len(train_df), len(val_df), len(test_df)


Label mapping (class -> id):
Eastern -> 0
Hijazi -> 1
Najdi -> 2
Southern -> 3


(800, 100, 100)

#Converting to Hugging Face Dataset Format


The Hugging Face `Trainer` API requires data to be in `Dataset` format rather than pandas DataFrames.

`Dataset.from_pandas()` converts each DataFrame split into a Hugging Face `Dataset` object, which:
- Supports efficient **batched tokenization** via `.map()`
- Is **memory-efficient**, using memory-mapped Arrow files
- Integrates directly with `Trainer` without any custom data loaders

We create three separate `Dataset` objects: `train_ds`, `val_ds`, and `test_ds`.

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df[["clean_text", "label"]])
val_ds   = Dataset.from_pandas(val_df[["clean_text", "label"]])
test_ds  = Dataset.from_pandas(test_df[["clean_text", "label"]])

train_ds


Dataset({
    features: ['clean_text', 'label', '__index_level_0__'],
    num_rows: 800
})

# Loading the AraBERT Baseline Model

We use **AraBERT v02** (`aubmindlab/bert-base-arabertv02`) as our baseline classifier. AraBERT is a BERT-based transformer model pre-trained on large-scale Arabic text corpora, making it well-suited for Arabic dialect understanding.

### Model Architecture
- **12 Transformer encoder layers** with hidden size 768 and 12 attention heads
- A **linear classification head** (softmax) is added on top with `num_labels=4` to predict one of four Saudi dialects
- No architectural modifications — this is a clean baseline

### Why AraBERT?
Standard multilingual models (like mBERT) have limited exposure to Arabic. AraBERT was trained specifically on Arabic and handles the **morphological richness**, **diacritics**, and **dialectal variations** of the language far better.

The model is moved to **GPU (`cuda`)** if available, otherwise CPU.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "aubmindlab/bert-base-arabertv02"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
device


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


'cuda'

#  Tokenizing the Classification Dataset

The `tokenize_batch_imp()` function applies the **AraBERT WordPiece tokenizer** to the cleaned text. The tokenizer performs:

- **Subword segmentation** — rare or dialectal words are broken into known subword units, enabling the model to handle unseen words
- **Padding** — all sequences are padded to `max_length=256` so batches have uniform shape
- **Truncation** — sequences longer than 256 tokens are cut off

The tokenizer outputs three tensors per sample:
- `input_ids` — numerical token indices
- `attention_mask` — 1 for real tokens, 0 for padding
- `token_type_ids` — segment separators (all 0 for single-sentence tasks)

After tokenization, the raw `clean_text` column is removed (no longer needed), and the datasets are set to **PyTorch tensor format** for direct use by the Trainer.

In [ ]:
def tokenize_batch_imp(batch):
    return tokenizer(
        batch["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_tok_imp = train_ds.map(tokenize_batch_imp, batched=True)
val_tok_imp   = val_ds.map(tokenize_batch_imp, batched=True)
test_tok_imp  = test_ds.map(tokenize_batch_imp, batched=True)

train_tok_imp = train_tok_imp.remove_columns(["clean_text"])
val_tok_imp   = val_tok_imp.remove_columns(["clean_text"])
test_tok_imp  = test_tok_imp.remove_columns(["clean_text"])

train_tok_imp.set_format("torch")
val_tok_imp.set_format("torch")
test_tok_imp.set_format("torch")

train_tok_imp[0].keys()


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

dict_keys(['label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'])

#  Defining Evaluation Metrics

This function is passed to the `Trainer` and is called automatically after each evaluation epoch.

Given the model's raw logit outputs and the true labels, it:
1. Converts logits to predicted class indices using `argmax`
2. Computes **Accuracy** — the proportion of correctly classified samples
3. Computes **Weighted F1-Score** — the harmonic mean of precision and recall, weighted by class frequency to account for any class imbalance

These two metrics together give a complete picture of how well the model distinguishes between the four Saudi dialects.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}


#  Configuring and Launching Baseline AraBERT Training

### - Training Configuration
| Hyperparameter | Value | Rationale |
|---|---|---|
| Epochs | 8 | Enough iterations for the model to converge on the relatively small dataset |
| Batch size | 16 | Fits comfortably in GPU memory while maintaining stable gradients |
| Learning rate | 2e-5 | Standard conservative rate for fine-tuning BERT-based models |
| Weight decay | 0.01 | L2 regularization to reduce overfitting |

> **Note:** `WANDB_DISABLED=true` suppresses Weights & Biases logging to keep the output clean.

### - Trainer
The Hugging Face `Trainer` handles the full training loop automatically, including forward/backward passes, gradient updates, evaluation at the end of each epoch, and checkpoint saving. This is the **baseline model** — full parameter fine-tuning with no optimization techniques applied.

In [ ]:
from transformers import TrainingArguments, Trainer

batch_size = 16
training_args_imp = TrainingArguments(
    output_dir="./arabert_saudial_baseline_improved",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    learning_rate=2e-5,
    weight_decay=0.01
)

import os
os.environ["WANDB_DISABLED"] = "true"

trainer = Trainer(
    model=model,
    args=training_args_imp,
    train_dataset=train_tok_imp,
    eval_dataset=val_tok_imp,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


trainer.train()

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-2626073215.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


TrainOutput(global_step=400, training_loss=0.9146957397460938, metrics={'train_runtime': 289.608, 'train_samples_per_second': 22.099, 'train_steps_per_second': 1.381, 'total_flos': 841970496307200.0, 'train_loss': 0.9146957397460938, 'epoch': 8.0})

# Evaluating the Baseline Classifier on the Test Set

After training, the model is evaluated on the **held-out test set** — data it has never seen during training or validation.

We generate:
- **Overall Accuracy** — percentage of correctly classified dialect samples
- **Weighted F1-Score** — robust metric that accounts for class-level imbalance
- **Classification Report** — per-class breakdown of Precision, Recall, and F1 for each of the four dialects (Najdi, Hijazi, Eastern, Southern)
- **Confusion Matrix** — shows where the model confuses one dialect for another; particularly informative for detecting which dialect pairs are hardest to distinguish

### - Expected Findings
The baseline achieves ~**85% accuracy** and ~**0.90 macro F1**, demonstrating that AraBERT captures region-specific linguistic patterns even on a small dataset. Misclassifications are most common between **Najdi and Hijazi**, which are linguistically closer than the other pairs.

In [ ]:
test_results = trainer.evaluate(test_tok_imp)


from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

pred_output = trainer.predict(test_tok_imp)
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=-1)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))
print("\nClassification report:\n")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

cm = confusion_matrix(y_true, y_pred)
cm

Accuracy: 0.45
Weighted F1: 0.4521675207888179

Classification report:

              precision    recall  f1-score   support

     Eastern       0.48      0.50      0.49        24
      Hijazi       0.55      0.41      0.47        27
       Najdi       0.34      0.41      0.37        27
    Southern       0.48      0.50      0.49        22

    accuracy                           0.45       100
   macro avg       0.46      0.45      0.45       100
weighted avg       0.46      0.45      0.45       100



array([[12,  1,  9,  2],
       [ 5, 11,  6,  5],
       [ 5,  6, 11,  5],
       [ 3,  2,  6, 11]])

# Measuring Baseline Training Time

This cell records the **wall-clock time** taken to train the baseline AraBERT model from start to finish.

- A `time.time()` snapshot is taken **before** and **after** `trainer.train()`
- The difference is the total training duration in seconds

This metric is essential for the **efficiency comparison** between the full fine-tuning baseline and the QLoRA-optimized model. A key expected outcome is that QLoRA significantly reduces training time by having far fewer trainable parameters.

>  Note: This triggers a second training run. If you want to preserve the first training results, you can skip re-running and instead log the time from the first `trainer.train()` call.

In [ ]:
import time
print("Starting Baseline Fine-Tuning...")
start_time = time.time() # Start the timer
trainer.train()
end_time = time.time() # Stop the timer
baseline_training_time = end_time - start_time
print(f"Baseline Training Time: {baseline_training_time:.2f} seconds")

Starting Baseline Fine-Tuning...


Step,Training Loss


# Measuring Peak GPU Memory (VRAM) Usage

This cell uses the **`pynvml`** library (NVIDIA Management Library) to query the GPU's memory usage.

The `get_peak_vram()` function:
1. Initializes the NVML interface
2. Gets a handle to GPU device 0
3. Reads the currently **used VRAM in GB**

We record VRAM usage **before and after** training, then take the maximum as the peak usage.

This measurement is critical for comparing efficiency: the baseline (full fine-tuning) loads and updates **all 110M+ AraBERT parameters** into VRAM, while QLoRA only updates a small fraction, dramatically reducing memory footprint.

In [ ]:
import pynvml

def get_peak_vram():
    """Returns the current VRAM usage in GB."""
    try:
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        return info.used / (1024**3)
    except:
        return 0.0
initial_vram = get_peak_vram()
final_vram = get_peak_vram()
baseline_peak_vram = max(initial_vram, final_vram)
print(f"Baseline Peak VRAM Usage: {baseline_peak_vram:.2f} GB")

Baseline Peak VRAM Usage: 4.64 GB


# Measuring the Saved Model Size on Disk

This cell calculates the total disk size of the saved baseline model checkpoint directory (`./arabert_saudial_baseline_improved`).

It iterates over all files in the checkpoint folder, sums their sizes in bytes, and converts to **gigabytes (GB)**.

For the full fine-tuned baseline, the saved checkpoint includes **all model weights** — making it comparatively large. In contrast, the QLoRA approach only saves the **adapter weights** (a tiny fraction of the original model size), which is one of its primary advantages in real-world deployment scenarios.

In [ ]:
import os

checkpoint_path = './arabert_saudial_baseline_improved'
total_size_bytes = sum(os.path.getsize(os.path.join(checkpoint_path, f))
                       for f in os.listdir(checkpoint_path)
                       if os.path.isfile(os.path.join(checkpoint_path, f)))
baseline_model_size_gb = total_size_bytes / (1024**3)

print(f"Baseline Model Size (on Disk): {baseline_model_size_gb:.2f} GB")

Baseline Model Size (on Disk): 0.00 GB


# Installing QLoRA-Specific Packages

Before applying the QLoRA optimization, we install the required libraries:

- **`bitsandbytes`** – Enables 4-bit quantization of model weights; required for QLoRA
- **`transformers==4.38.1`** – Pinned version for stability with PEFT/bitsandbytes integration
- **`peft==0.9.0`** – Parameter-Efficient Fine-Tuning library; provides `LoraConfig` and `get_peft_model`
- **`accelerate==0.27.2`** – Required for mixed-precision and distributed training

> Pinned versions are intentional — these specific combinations are known to work together for QLoRA on Colab. Upgrading any package may cause incompatibilities.

In [ ]:
!pip install -U bitsandbytes
!pip install transformers==4.38.1 peft==0.9.0 accelerate==0.27.2 datasets scikit-learn pynvml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 18.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 100.5 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  

# Optimized Classification Using QLoRA

This cell applies **QLoRA (Quantized Low-Rank Adaptation)** — a Parameter-Efficient Fine-Tuning (PEFT) technique — to the same AraBERT classification task. QLoRA is the core optimization of this project.

## How QLoRA Works
Instead of updating all 110M+ model parameters (as in full fine-tuning), QLoRA:

1. **Quantizes** the frozen base model weights to **4-bit precision** using `BitsAndBytesConfig` (type: NF4 — NormalFloat4, with double quantization for extra compression)
2. **Freezes** all original model weights — they are never updated
3. **Injects trainable low-rank adapter matrices** (LoRA) into the attention layers (`query`, `key`, `value`)
4. **Only the adapter weights** are updated during training — a tiny fraction of total parameters

## LoRA Configuration
| Parameter | Value | Meaning |
|---|---|---|
| `r` (rank) | 8 | Size of the low-rank decomposition matrices |
| `lora_alpha` | 16 | Scaling factor for LoRA updates |
| `target_modules` | query, key, value | Attention projection layers where adapters are injected |
| `lora_dropout` | 0.05 | Regularization within the adapters |

## Efficiency Gains
| Metric | Baseline | QLoRA |
|---|---|---|
| Trainable parameters | 110M+ (100%) | ~1% |
| GPU memory | High | Significantly reduced |
| Saved model size | Full weights (~GB) | Adapter only (~MB) |

## Training & Evaluation
The QLoRA model is trained for 8 epochs with the same dataset splits. After training, the adapter is saved to `./qlora_adapter_classification`, and test accuracy, F1, training time, VRAM, and adapter size are all reported.

In [ ]:
import torch
import numpy as np
import time
import os
import pynvml
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Re-install bitsandbytes and restart runtime to ensure the latest version is loaded
!pip install -U bitsandbytes
import os
os.kill(os.getpid(), 9)

MODEL_NAME = "aubmindlab/bert-base-arabertv02"
NUM_LABELS = 4

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    f1_weighted = f1_score(labels, predictions, average="weighted")
    return {"accuracy": accuracy, "f1_weighted": f1_weighted}

def get_gpu_usage():
    try:
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        return info.used / (1024**3)
    except Exception as e:
        print(f"Warning: GPU monitoring failed ({e})")
        return 0.0
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_use_double_quant=True,
)
print("Loading model with 4-bit Quantization (QLoRA base)")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    quantization_config=bnb_config,
)
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "key", "value"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
)
model = get_peft_model(model, lora_config)
print("\n--- Model Trainable Parameters ---")
model.print_trainable_parameters()
training_args = TrainingArguments(
    output_dir="./qlora_results",
    num_train_epochs=8,
    per_device_train_batch_size=8,
    learning_rate=2e-4,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    logging_steps=1,
    bf16=True,
    fp16=False,
    report_to="none",
    max_grad_norm=0.3,
    load_best_model_at_end=True,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok_imp,
    eval_dataset=val_tok_imp,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("\nStarting QLoRA Fine-Tuning...")
start_time = time.time()
start_vram = get_gpu_usage()
train_result = trainer.train()
end_time = time.time()
end_vram = get_gpu_usage()
qlora_training_time = end_time - start_time
test_results = trainer.evaluate(test_tok_imp)
adapter_path = "./qlora_adapter_classification"
model.save_pretrained(adapter_path)
adapter_size_mb = os.path.getsize(f"{adapter_path}/adapter_model.safetensors") / (1024**2)

print(" QLoRA CLASSIFICATION OPTIMIZATION RESULTS ")
print(f"1. Test Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"2. Test Weighted F1: {test_results['eval_f1_weighted']:.4f}")
print("-" * 45)
print(f"3. QLoRA Training Time: {qlora_training_time:.2f} seconds")
print(f"4. QLoRA Peak VRAM Usage: {max(start_vram, end_vram):.2f} GB")
print(f"5. QLoRA Model Size (Adapter Only): {adapter_size_mb:.2f} MB")

Loading model with 4-bit Quantization (QLoRA base)


ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

# Preparing Data for Task 2: Dialect-to-MSA Translation

This section begins the **second NLP task**: translating Saudi dialect text into **Modern Standard Arabic (MSA)** using a sequence-to-sequence model.

We extract two columns from the SauDial dataset:
- **`Dialect Translation`** → renamed to `source_text` (the dialect input)
- **`Modern Standard Arabic (MSA) Translation`** → renamed to `target_text` (the MSA output)

Rows with missing values in either column are dropped using `.dropna()`.

The resulting `df_trans` DataFrame forms a **parallel corpus** — each row is a (dialect, MSA) pair suitable for supervised seq2seq training, similar to how neural machine translation datasets are structured.

In [ ]:
# Extracting the relevant translation columns from SauDial.
df_trans = df[[
    "Dialect Translation",
    "Modern Standard Arabic (MSA) Translation"
]].dropna()

# Renaming columns for clarity and consistency
df_trans = df_trans.rename(columns={
    "Dialect Translation": "source_text",                         # Dialect text input
    "Modern Standard Arabic (MSA) Translation": "target_text"     # MSA translation output
})

# Displaying the first few rows to verify correct extraction
df_trans.head()


,source_text,target_text
0,"""قوّوا مكاينكم! سباق اليوم بياخذكم على دروب ال...","""شغلوا محركاتكم! سباق اليوم يأخذكم عبر طرق الت..."
1,"""انت محبوس في الدور الـ300 من برج جدة. حل الأل...","""أنت محاصر في الطابق الثلاثمائة من برج جدة. حل..."
2,"""جدي يقول: 'البوم المضبوط يقدر يواجه أي نوة.' ...","""يقول جدي: 'السفينة المبنية جيدًا يمكنها مواجه..."
3,"""المطر جاي بدري السنة. لازم نقوي المساطب ونزرع...","""الأمطار تأتي مبكرًا هذا العام. نحتاج إلى تقوي..."
4,"""حياكم في مسابقة زين الهجن الملكية! حكموا على ...","""مرحبًا بكم في مسابقة جمال الإبل الملكية! احكم..."


# Splitting the Translation Dataset

The translation dataset is divided into three splits using the same 80/10/10 ratio as the classification task:

- **`train_trans`** (80%) — used to train the seq2seq model
- **`val_trans`** (10%) — used for per-epoch evaluation during training
- **`test_trans`** (10%) — held out for final BLEU/ROUGE evaluation

`random_state=42` ensures the split is reproducible. Note that unlike the classification split, no stratification is applied here since there are no discrete class labels — this is a generation task.

In [ ]:
from sklearn.model_selection import train_test_split

train_trans, temp_trans = train_test_split(
    df_trans,
    test_size=0.2,
    random_state=42
)

val_trans, test_trans = train_test_split(
    temp_trans,
    test_size=0.5,
    random_state=42
)

len(train_trans), len(val_trans), len(test_trans)


(800, 100, 100)

# Converting Translation DataFrames to Hugging Face Dataset Format

Just as with the classification task, we convert the pandas DataFrames into Hugging Face `Dataset` objects for compatibility with the `Seq2SeqTrainer`.

The printed output shows the **schema** of the training dataset — confirming the `source_text` and `target_text` columns are present along with the total number of training samples.

In [ ]:
# Displaying the training dataset structure.
from datasets import Dataset

train_ds_trans = Dataset.from_pandas(train_trans)
val_ds_trans = Dataset.from_pandas(val_trans)
test_ds_trans = Dataset.from_pandas(test_trans)

train_ds_trans


Dataset({
    features: ['source_text', 'target_text', '__index_level_0__'],
    num_rows: 800
})

# Loading the Baseline Translation Model (mT5-small)

We use **`google/mt5-small`** — a multilingual T5 variant — as the seq2seq backbone for the dialect-to-MSA translation task.

### Why mT5?
- mT5 is an **encoder-decoder** transformer (unlike BERT which is encoder-only), making it natively suited for **text generation** tasks like translation
- The `small` variant has ~300M parameters — manageable on Colab with a T4 GPU
- It was pre-trained on 101 languages including Arabic, giving it a reasonable starting representation of Arabic text

The model and its tokenizer are loaded from Hugging Face Hub and moved to the available device (GPU or CPU).

In [ ]:
# Displaying which device is being used.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name_trans = "google/mt5-small"

tokenizer_trans = AutoTokenizer.from_pretrained(model_name_trans)
model_trans = AutoModelForSeq2SeqLM.from_pretrained(model_name_trans)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_trans = model_trans.to(device)

device


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:550: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

'cuda'

#  Defining the Tokenization Function for Seq2Seq Training

The `tokenize_translation()` function prepares both the **source (dialect)** and **target (MSA)** text for seq2seq training.

Key steps:
- **Source text** is tokenized into `input_ids` and `attention_mask` (standard encoder inputs)
- **Target text** is tokenized separately and its `input_ids` are stored as `labels` — what the decoder is trained to predict
- Both source and target are **padded and truncated** to `max_length=128` tokens for uniform batch shapes

The function is designed to be applied **in batches** via Hugging Face's `.map()` method for efficient processing.

In [ ]:
# Tokenizing the target text & assigning token IDs as labels.
# Returning a dictionary compatible with seq2seq training requirements.
def tokenize_translation(batch):
    inputs = tokenizer_trans(
        batch["source_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

    targets = tokenizer_trans(
        batch["target_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

    inputs["labels"] = targets["input_ids"]
    return inputs


# Applying Tokenization Across All Translation Splits

The tokenization function is applied to all three dataset splits (train, validation, test) using `.map(batched=True)` for speed.

After tokenization:
- The original **raw text columns** (`source_text`, `target_text`) are removed — the model only needs the numerical token IDs
- All datasets are converted to **PyTorch tensor format** so they can be directly fed into the model during training and evaluation

In [ ]:
# Applying the tokenization function to all dataset splits.
# Removing the original text columns after tokenization to reduce memory usage.
# Converting the datasets into PyTorch tensor format.

train_tok_trans = train_ds_trans.map(tokenize_translation, batched=True)
val_tok_trans = val_ds_trans.map(tokenize_translation, batched=True)
test_tok_trans = test_ds_trans.map(tokenize_translation, batched=True)

cols_to_remove = ["source_text", "target_text"]

train_tok_trans = train_tok_trans.remove_columns(cols_to_remove)
val_tok_trans = val_tok_trans.remove_columns(cols_to_remove)
test_tok_trans = test_tok_trans.remove_columns(cols_to_remove)

train_tok_trans.set_format("torch")
val_tok_trans.set_format("torch")
test_tok_trans.set_format("torch")


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

# Configuring Training Arguments for Baseline Translation

We define the training hyperparameters for the **mT5-small baseline translation model** using `Seq2SeqTrainingArguments`:

| Parameter | Value | Rationale |
|---|---|---|
| Epochs | 5 | Standard starting point for seq2seq fine-tuning |
| Batch size | 8 | Smaller than classification due to the larger seq2seq model memory footprint |
| Learning rate | 3e-4 | Slightly higher than classification; seq2seq models benefit from a larger LR |
| Weight decay | 0.01 | Regularization to reduce overfitting |
| Evaluation & Save strategy | per epoch | Saves the best checkpoint based on validation loss |

This is **full-parameter fine-tuning** with no efficiency optimizations — the baseline against which QLoRA will later be compared.

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# Defining training arguments using Seq2SeqTrainingArguments.

training_args_trans = Seq2SeqTrainingArguments(
    output_dir="./mt5_baseline_translation",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=3e-4,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    report_to="none"
)


# Instantiating the Seq2SeqTrainer

The `Seq2SeqTrainer` is the Hugging Face training loop for encoder-decoder models. It is a specialized version of `Trainer` that handles the sequence-to-sequence-specific data flow (encoder input → decoder output).

It takes:
- The **mT5-small model** as the backbone
- The **training arguments** defined above
- The **tokenized train and validation datasets**
- The **mT5 tokenizer** for padding/decoding during evaluation

No custom loss function is needed — the Seq2SeqTrainer handles cross-entropy over token sequences by default.

In [ ]:
# Creating the Seq2SeqTrainer for translation fine-tuning.

trainer_trans = Seq2SeqTrainer(
    model=model_trans,
    args=training_args_trans,
    train_dataset=train_tok_trans,
    eval_dataset=val_tok_trans,
    tokenizer=tokenizer_trans
)


# Training the Baseline Translation Model

This cell launches the full fine-tuning of the **mT5-small baseline translation model**.

Training time is measured using `time.time()` before and after `trainer.train()`. This serves as the **efficiency reference point** — after training, all 289M+ parameters of mT5-small will have been updated.

### Expected Outcome
Given the small dataset size (~640 training samples) and the complexity of dialect-to-MSA translation, the baseline model is expected to **struggle** — generating short, incomplete, or incoherent outputs. This poor baseline performance motivates the use of QLoRA optimization, which dramatically improves results by enabling more targeted, efficient fine-tuning.

In [ ]:
import time

print("Starting Baseline Translation Fine-Tuning...")
start_time_base_trans = time.time()

trainer_trans.train()

end_time_base_trans = time.time()
baseline_translation_training_time = end_time_base_trans - start_time_base_trans
print(f"Baseline Translation Training Time: {baseline_translation_training_time:.2f} seconds")


Epoch,Training Loss,Validation Loss
1,18.506700,5.292872
2,4.850600,2.436249
3,2.751900,2.012480
4,2.514400,1.941166
5,2.438000,1.939133


TrainOutput(global_step=500, training_loss=6.212336151123047, metrics={'train_runtime': 584.1737, 'train_samples_per_second': 6.847, 'train_steps_per_second': 0.856, 'total_flos': 528749690880000.0, 'train_loss': 6.212336151123047, 'epoch': 5.0})

## Optimized Translation with QLoRA (mT5-small)

In [ ]:
!pip install evaluate
!pip install rouge_score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=f4e98c71fcffbaf9e47fdfe47319cdd7636aafbb68018e3351cef67af6497bd1
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


# Installing Translation Evaluation Libraries

We install two additional packages needed to evaluate the translation output:

- **`evaluate`** – Hugging Face's unified evaluation library; provides standard NLP metrics
- **`rouge_score`** – Backend dependency required by the `evaluate` library to compute ROUGE scores

# Evaluating the Baseline Translation Model

This cell evaluates the quality of the baseline mT5-small model's translations on the **test set** using industry-standard machine translation metrics.

### Evaluation Process
For each sample in the test set:
1. The **dialect source text** is tokenized and passed through the model
2. The model **generates** a translation using `model.generate()`
3. The generated output is **decoded back to text** using the tokenizer
4. Generated translations are compared against the reference MSA texts

### Metrics Used
| Metric | What it Measures |
|---|---|
| **BLEU** | N-gram precision overlap between predicted and reference translations. Scores range 0–1; higher is better |
| **BLEU 1-gram / 2-gram / 3-gram / 4-gram** | Progressively stricter n-gram matching — reveals where precision breaks down |
| **Brevity Penalty** | Penalizes outputs that are shorter than the reference |
| **ROUGE-1 / ROUGE-2 / ROUGE-L** | Recall-based overlap metrics; ROUGE-L measures longest common subsequence |
| **Perplexity** | Language fluency measure — reported as `None` due to environment limitations |

### Expected Results (Baseline)
The baseline is expected to produce near-zero BLEU and ROUGE scores. This is because:
- The model generates **short, incomplete outputs** (triggering a low brevity penalty)
- Full fine-tuning of a large model on a small dataset leads to **poor generalization**
- These poor results set the reference point that QLoRA optimization dramatically improves upon (BLEU: 0.0 → 0.775, ROUGE-1: 0.0 → 0.814)

In [ ]:
# Evaluating the baseline translation model using the evaluate library.
# Generating predictions manually and computing BLEU and ROUGE scores.

import evaluate
import math

# Loading BLEU and ROUGE metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

pred_text = []
true_text = []

# Generating translations
for sample in test_ds_trans:

    # Tokenizing source sentence
    inputs = tokenizer_trans(
        sample["source_text"],
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    ).to(device)

    # Generating translation
    output_tokens = model_trans.generate(
        **inputs,
        max_length=128
    )

    predicted_sentence = tokenizer_trans.decode(output_tokens[0], skip_special_tokens=True)
    reference_sentence = sample["target_text"]

    pred_text.append(predicted_sentence)
    true_text.append(reference_sentence)

# BLEU
bleu_score = bleu.compute(
    predictions=pred_text,
    references=[[t] for t in true_text]
)

# ROUGE
rouge_score = rouge.compute(
    predictions=pred_text,
    references=true_text
)

perplexity = None

bleu_score, rouge_score, perplexity


({'bleu': 0.0,
  'precisions': [0.06818181818181818, 0.02167630057803468, 0.0, 0.0],
  'brevity_penalty': 0.15589292921179554,
  'length_ratio': 0.3498233215547703,
  'translation_length': 792,
  'reference_length': 2264},
 {'rouge1': np.float64(0.0),
  'rouge2': np.float64(0.0),
  'rougeL': np.float64(0.0),
  'rougeLsum': np.float64(0.0)},
 None)